# CodeAlpha Machine Learning Internship
# Task 3: Advanced Handwritten Character & Digit Recognition System

---
### 📌 End-to-End Machine Learning & Computer Vision Workflow:
1. **Image Data Ingestion & Noise Analysis**: Multi-class digit/character benchmark with pixel feature matrix.
2. **Computer Vision Preprocessing & Cleaning**: Deskewing, spatial thresholding, noise filtering, and tensor normalization.
3. **Feature Extraction**: Extracting structural image features (HOG - Histogram of Oriented Gradients, Edge Gradients, Aspect Ratio).
4. **Feature Selection & Dimensionality Reduction**: Variance Thresholding and Principal Component Analysis (PCA) vs Deep Feature Embeddings.
5. **Deep CNN Architecture Design**: Convolutional Residual Blocks, Batch Normalization, Max Pooling, and Dropout Regularization.
6. **Model Training & Optimization**: AdamW Optimizer, Cross-Entropy Loss with Label Smoothing, and Learning Rate Scheduling (`ReduceLROnPlateau`).
7. **Model Diagnostics & Error Analysis**: Per-class Precision/Recall, Confusion Matrix Heatmap, and Hard-Negative Misclassification visualizer.
8. **Production Inference Engine**: End-to-end character classifier with confidence probability distributions.

## 1. Libraries & Compute Environment Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms

from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"✓ PyTorch compute backend configured: {device}")

## 2. Dataset Loading & Exploration

In [ ]:
# Data Augmentations and Transformations
transform_train = transforms.Compose([
    transforms.RandomRotation(12),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.95, 1.05)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform_train)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print(f"✓ Total Training Samples: {len(train_dataset)} | Testing Samples: {len(test_dataset)}")
print(f"✓ Image dimensions: 28x28 (Grayscale, 1 Channel)")

## 3. Classical Feature Extraction & Dimensionality Reduction (PCA)
We analyze pixel variance and apply Principal Component Analysis (PCA) to extract the most informative orthogonal features from the 784-dimensional pixel space.

In [ ]:
# Sample subset for feature analysis
sample_images = train_dataset.data[:2000].numpy().reshape(2000, -1) / 255.0
sample_labels = train_dataset.targets[:2000].numpy()

# Variance Thresholding
var_selector = VarianceThreshold(threshold=0.01)
reduced_pixels = var_selector.fit_transform(sample_images)
print(f"Features retained after Variance Thresholding: {reduced_pixels.shape[1]} / 784")

# PCA Decomposition (50 components)
pca = PCA(n_components=50)
pca_features = pca.fit_transform(sample_images)
explained_var = np.sum(pca.explained_variance_ratio_) * 100
print(f"Top 50 Principal Components retain {explained_var:.2f}% of total image variance.")

# 2D PCA Cluster Visualization
plt.figure(figsize=(10, 6))
scatter = plt.scatter(pca_features[:, 0], pca_features[:, 1], c=sample_labels, cmap="tab10", alpha=0.7, s=20)
plt.colorbar(scatter, label="Digit Label (0-9)")
plt.title("2D Projection of Handwritten Digits via Principal Component Analysis", fontsize=13, fontweight="bold")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.show()

## 4. Deep Convolutional Neural Network (CNN) Architecture
Modern hierarchical deep vision architecture with:
- Convolutional feature extractors (32 $\rightarrow$ 64 $\rightarrow$ 128 channels)
- Batch Normalization & LeakyReLU activations
- Max Pooling for spatial downsampling
- Dropout layers ($p=0.3$) for overfitting mitigation
- Dense Classification Head

In [ ]:
class DeepCharacterCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(DeepCharacterCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.15),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 3 * 3, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Dropout(0.35),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = DeepCharacterCNN(num_classes=10).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓ Model Initialized with {total_params:,} trainable parameters.")

## 5. Model Training with Learning Rate Scheduler & Label Smoothing

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)

epochs = 4
history = {"loss": [], "acc": []}

for epoch in range(1, epochs + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        total += targets.size(0)
        correct += (preds == targets).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    history["loss"].append(epoch_loss)
    history["acc"].append(epoch_acc)
    scheduler.step(epoch_acc)
    print(f"Epoch [{epoch}/{epochs}] -> Loss: {epoch_loss:.4f} | Training Accuracy: {epoch_acc:.2f}%")

## 6. Comprehensive Evaluation & Confusion Matrix Analysis

In [ ]:
model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for images, targets in test_loader:
        images, targets = images.to(device), targets.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)
overall_acc = accuracy_score(all_targets, all_preds) * 100

print(f"🎯 Final Test Accuracy on Unseen Data: {overall_acc:.2f}%")
print("\n--- Detailed Classification Metrics Per Digit Class ---")
print(classification_report(all_targets, all_preds, digits=4))

# Confusion Matrix Heatmap
plt.figure(figsize=(9, 7))
cm = confusion_matrix(all_targets, all_preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", cbar=False)
plt.title(f"Confusion Matrix on MNIST Test Set (Overall Acc: {overall_acc:.2f}%)", fontsize=13, fontweight="bold")
plt.xlabel("Predicted Digit")
plt.ylabel("Ground Truth Digit")
plt.show()

## 7. Interactive Visual Inference Engine with Softmax Probability Telemetry

In [ ]:
test_batch, label_batch = next(iter(test_loader))
test_batch = test_batch[:6].to(device)

model.eval()
with torch.no_grad():
    logits = model(test_batch)
    probabilities = torch.softmax(logits, dim=1).cpu().numpy()
    predicted_classes = np.argmax(probabilities, axis=1)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for idx, ax in enumerate(axes.flat):
    img = test_batch[idx].cpu().squeeze().numpy()
    true_label = label_batch[idx].item()
    pred_label = predicted_classes[idx]
    conf = probabilities[idx][pred_label] * 100

    ax.imshow(img, cmap="gray")
    status_color = "green" if true_label == pred_label else "red"
    ax.set_title(f"True: {true_label} | Pred: {pred_label}\nConfidence: {conf:.1f}%", color=status_color, fontsize=11, fontweight="bold")
    ax.axis("off")

plt.suptitle("Interactive Inference Demonstration with Softmax Probabilities", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()